# 实验报告 03　正则表达式与网络结构化数据

姓名：欧阳洋　　学号：20244080124

In [1]:
import hashlib
import os
import platform
import sys
import uuid

STUDENT_ID = 20244080124
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")
print(f"[Python]    {sys.version.split()[0]}")

[Conda 环境] python_work1
[学号]       20244080124
[机器指纹] 66ab728912afd30e
[RUN_ID]   20244080124-66ab728912afd30e
[Python]    3.12.15


## 实验 1　正则表达式的基础匹配

### 1.1 数字、空白和单词边界

`\d+` 匹配连续数字，`\s+` 匹配连续空白。`\bcat\b` 要求两边都是单词边界，
因此只命中单独的 `cat`。题给文本里没有数字，数字匹配返回空列表。

In [2]:
import re

text = "cat on mat; later: cats and dogs"
print("原始文本：", text)
for pattern in (r"\d+", r"\s+", r"\bcat\b"):
    print(pattern, re.findall(pattern, text))

原始文本： cat on mat; later: cats and dogs
\d+ []
\s+ [' ', ' ', ' ', ' ', ' ', ' ']
\bcat\b ['cat']


### 1.2 交替匹配与分支顺序

`cat|dog` 匹配 `cat` 或 `dog`，默认区分大小写。在同一个起始位置，正则引擎按顺序尝试各分支，
采用第一个成功的匹配。`a` 在前时得到四个 `a`，`aaa` 在前时先匹配 `aaa`，剩下一个 `a`。

In [3]:
print("cat|dog：", re.findall(r"cat|dog", "cat dog bird CAT"))
print("a|aa|aaa：", re.findall(r"a|aa|aaa", "aaaa"))
print("aaa|aa|a：", re.findall(r"aaa|aa|a", "aaaa"))

cat|dog： ['cat', 'dog']
a|aa|aaa： ['a', 'a', 'a', 'a']
aaa|aa|a： ['aaa', 'a']


### 1.3 比较量词

这五个模式的量词只限定 `a` 出现的次数。

| 模式 | `a` 的允许次数 |
| --- | --- |
| `ca*t` | 零次或多次 |
| `ca+t` | 一次或多次 |
| `ca?t` | 零次或一次 |
| `ca{2}t` | 恰好两次 |
| `ca{1,2}t` | 一次或两次 |

In [4]:
text = "ct cat caat caaat"
for pattern in (r"ca*t", r"ca+t", r"ca?t", r"ca{2}t", r"ca{1,2}t"):
    print(pattern, re.findall(pattern, text))

ca*t ['ct', 'cat', 'caat', 'caaat']
ca+t ['cat', 'caat', 'caaat']
ca?t ['ct', 'cat']
ca{2}t ['caat']
ca{1,2}t ['cat', 'caat']


## 实验 2　分组、回溯引用与替换

### 2.1 邮箱与重复单词

邮箱模式分别捕获 `@` 前的用户名和 `@` 后的域名。两个捕获组使 `findall()` 为每条匹配返回一个二元组。
重复词模式中的 `\1` 引用第一个组匹配到的单词，两个相同单词之间允许一个或多个空白字符。

In [5]:
contacts = "联系：alice@example.com；备选：bob@test.org"
print("邮箱（用户名、域名）：", re.findall(r"(\w+)@(\w+\.\w+)", contacts))

words = "cat cat, dog cat, bird bird"
print("连续重复的单词：", re.findall(r"\b(\w+)\s+\1\b", words))

邮箱（用户名、域名）： [('alice', 'example.com'), ('bob', 'test.org')]
连续重复的单词： ['cat', 'bird']


### 2.2 统一日期与电话格式

日期模式只替换匹配到的日期，将年、月、日三个捕获组按 `\2-\3-\1` 重排，原有换行保留。
电话模式依次捕获三位区号、三位前缀和四位尾号，再用括号、空格与连字符重排。

In [6]:
dates = "2024-05-01\n2024/05/02\n2024.05.03"
converted_dates = re.sub(r"(\d{4})[-/.](\d{2})[-/.](\d{2})", r"\2-\3-\1", dates)
print("转换前：\n" + dates)
print("转换后：\n" + converted_dates)
print(f"行数：{len(dates.splitlines())} → {len(converted_dates.splitlines())}")

phones = "联系我：123-456-7890 或 987.654.3210"
converted_phones = re.sub(r"(\d{3})[-.](\d{3})[-.](\d{4})", r"(\1) \2-\3", phones)
print("原始电话：", phones)
print("统一格式：", converted_phones)

转换前：
2024-05-01
2024/05/02
2024.05.03
转换后：
05-01-2024
05-02-2024
05-03-2024
行数：3 → 3
原始电话： 联系我：123-456-7890 或 987.654.3210
统一格式： 联系我：(123) 456-7890 或 (987) 654-3210


## 实验 3　使用 BeautifulSoup 解析 HTML

### 3.1 构建解析树并读取标题

将题给 HTML 字符串传给 `html.parser`。通过 `<title>` 获取文档标题，通过 `<h1>` 获取正文一级标题。

In [7]:
from bs4 import BeautifulSoup

html_doc = """
<html><head><title>实验课程</title></head><body>
<h1>数据分析</h1>
<p class="course">第六讲：正则表达式</p>
<p class="course">第七讲：结构化数据</p>
<a href="https://example.org/lecture06">讲义六</a>
<a href="https://example.org/lecture07">讲义七</a>
</body></html>
"""

soup = BeautifulSoup(html_doc, "html.parser")


title = soup.find("title")
h1 = soup.find("h1")
print("页面标题：", title.get_text())
print("一级标题的标签名与文本：", h1.name, h1.get_text())

页面标题： 实验课程
一级标题的标签名与文本： h1 数据分析


### 3.2 查找课程段落与链接

用 `find_all()` 按标签名和 `class` 属性筛选两段课程文字，再逐一读取两个链接的文本和 `href` 属性。

In [8]:
for paragraph in soup.find_all("p", class_="course"):
    print("课程段落：", paragraph.get_text())

for link in soup.find_all("a"):
    print("链接文字与地址：", link.get_text(), link["href"])

课程段落： 第六讲：正则表达式
课程段落： 第七讲：结构化数据
链接文字与地址： 讲义六 https://example.org/lecture06
链接文字与地址： 讲义七 https://example.org/lecture07


### 3.3 比较文本提取范围

整个文档的 `get_text()` 包含 `<title>` 和正文的文字，保留文本节点里的换行。
对单个段落或链接调用时，只提取选中节点范围内的文字。链接地址需单独读取 `href`。

In [9]:
print("整个文档：", repr(soup.get_text()))
print("首个段落：", repr(soup.find("p").get_text()))
print("首个链接：", repr(soup.find("a").get_text()))

整个文档： '\n实验课程\n数据分析\n第六讲：正则表达式\n第七讲：结构化数据\n讲义六\n讲义七\n\n'
首个段落： '第六讲：正则表达式'
首个链接： '讲义六'
